In [1]:
import pandas as pd
from prophet import Prophet
import numpy as np
import matplotlib.pyplot as plt


C:\python_packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


In [ ]:
df = pd.read_csv('data\wdi_fertility_data_filled_filtered.csv')

In [ ]:
df.tail()

In [ ]:
df.year.unique()

In [ ]:
countries =  df['country_name'].unique()
results = []
var = 'total_fertility_rate'
regressors = ['gdp_per_capita_2015_dollar','physicians_per_1000', 'hospital_beds_per_1000','crop_production_index', 'life_expectancy_at_birth']
forecast_years = 5

### Prophet

In [ ]:
for country in countries:
    country_data = df[df['country_name'] == country].sort_values(by='year')
    #wydzielenie zbioru train
    #country_data = country_data[country_data['year'] <= 2015]
    max_year = country_data['year'].max() 
    
    
    prophet_data = country_data[['year', var] +  regressors].rename(columns={'year': 'ds', var: 'y'})
    prophet_data['ds'] = pd.to_datetime(prophet_data['ds'], format='%Y') 

    # regresor
    first_reg = 0
    for regressor in regressors:
        regressor_model = Prophet()
        regressor_model.fit(prophet_data[['ds', regressor]].rename(columns={regressor: 'y'}))
    
        future_regressor = regressor_model.make_future_dataframe(periods=forecast_years + 1, freq='Y')
        regressor_forecast = regressor_model.predict(future_regressor)
    
        # dodanie predykcji regresora
        if first_reg == 0:
            future_regressor_values = regressor_forecast[['ds', 'yhat']].rename(columns={'yhat': regressor})
            first_reg = 1
        else:
            future_regressor_values[regressor] = regressor_forecast[['yhat']].rename(columns={'yhat': regressor})

    
    # Prophet
    prophet_model = Prophet()
    for regressor in regressors:
        prophet_model.add_regressor(regressor)
    prophet_model.fit(prophet_data)


    future = prophet_model.make_future_dataframe(periods=forecast_years + 1, freq='Y')
    future = future.merge(future_regressor_values, on='ds', how='left')
    forecast = prophet_model.predict(future)

    # progonzy
    forecast_next_years = forecast[forecast['ds'].dt.year > max_year]
    for _, row in forecast_next_years.iterrows():
        results.append({
            'country_name': country,
            'year': row['ds'].year,
            f'{var}_Prophet': row['yhat'],
            f'{var}_Lower_CI': row['yhat_lower'],
            f'{var}_Upper_CI': row['yhat_upper'],
        })

forecast_df = pd.DataFrame(results)



In [ ]:
future

In [ ]:

def plot_forecasts(df, forecast_df, country, var):
    country_data = df[df['country_name'] == country].sort_values(by='year')
    country_forecast = forecast_df[forecast_df['country_name'] == country].sort_values(by='year')

    plt.figure(figsize=(12, 6))
    plt.plot(country_data['year'], country_data[var], label='Dane historyczne', marker='o', color='blue')
    plt.plot(country_forecast['year'], country_forecast[f'{var}_Prophet'], label='Prognoza', marker='o', color='green')
    plt.fill_between(
        country_forecast['year'],
        country_forecast[f'{var}_Lower_CI'],
        country_forecast[f'{var}_Upper_CI'],
        color='lightgreen',
        alpha=0.5,
        label='Przedziały ufności'
    )

    plt.title(f'{country}')
    plt.xlabel('Rok')
    plt.ylabel(var)
    plt.axvline(x=country_data['year'].max(), color='gray', linestyle='--')
    plt.legend()
    plt.grid()
    plt.ylim(0, 7)
    plt.show()



In [ ]:
for country in countries:
    plot_forecasts(df, forecast_df, country, var='total_fertility_rate')
